# Second-Level Analysis: One-Sample t-Test

Group-level analysis testing whether the Switch > Stay contrast is significantly different from zero across all 75 subjects.

## 1. Load All First-Level Z-Maps

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import norm
from nilearn.glm.second_level import SecondLevelModel, non_parametric_inference
from nilearn.image import get_data, math_img
from nilearn.plotting import plot_glass_brain, plot_stat_map

results_dir = Path("../results")
effect_map_paths = sorted(results_dir.glob("sub-*_switch-vs-stay_effect.nii.gz"))
effect_maps = [str(p) for p in effect_map_paths]

print(f"Found {len(effect_maps)} subject effect maps (already in MNI152 2mm space)")

## 2. Set Up and Fit the Second-Level Model

In [ ]:
# Design matrix: column of ones for one-sample t-test (intercept only)
design_matrix = pd.DataFrame(
    [1] * len(effect_maps),
    columns=["intercept"],
)

print(f"Design matrix shape: {design_matrix.shape}")
design_matrix.head()

In [ ]:
second_level_model = SecondLevelModel(smoothing_fwhm=8.0, n_jobs=2)
second_level_model = second_level_model.fit(
    effect_maps,
    design_matrix=design_matrix,
)
print("Second-level model fitted successfully.")

## 3. Compute Group-Level Contrast

In [ ]:
z_map_group = second_level_model.compute_contrast(
    second_level_contrast="intercept",
    output_type="z_score",
)
print(f"Group z-map shape: {z_map_group.shape}")

## 4. Visualize Parametric Results

In [ ]:
# Threshold at p < 0.001 uncorrected
p001_unc = norm.isf(0.001)
print(f"Z threshold for p < 0.001: {p001_unc:.2f}")

plot_glass_brain(
    z_map_group,
    threshold=p001_unc,
    colorbar=True,
    display_mode="lyrz",
    plot_abs=False,
    title="Switch > Stay (group, p < 0.001 unc)",
)
plt.show()

plot_stat_map(
    z_map_group,
    threshold=p001_unc,
    title="Switch > Stay (group, p < 0.001 unc)",
    cut_coords=5,
    display_mode="z",
)
plt.show()

## 5. Bonferroni-Corrected p-Values

In [ ]:
p_val = second_level_model.compute_contrast(output_type="p_value")

n_voxels = np.sum(get_data(second_level_model.masker_.mask_img_))
print(f"Number of voxels in mask: {n_voxels}")

neg_log_pval = math_img(
    f"-np.log10(np.minimum(1, img * {n_voxels!s}))",
    img=p_val,
)

plot_glass_brain(
    neg_log_pval,
    colorbar=True,
    display_mode="lyrz",
    plot_abs=False,
    title="Switch > Stay (-log10 Bonferroni p-value)",
)
plt.show()

## 6. Non-Parametric Inference (Permutation Test)

In [ ]:
out_dict = non_parametric_inference(
    effect_maps,
    design_matrix=design_matrix,
    model_intercept=True,
    n_perm=500,
    two_sided_test=False,
    smoothing_fwhm=8.0,
    n_jobs=2,
    threshold=0.001,
)
print("Permutation test complete.")
print(f"Output keys: {list(out_dict.keys())}")

In [ ]:
plot_glass_brain(
    out_dict["logp_max_t"],
    colorbar=True,
    display_mode="lyrz",
    plot_abs=False,
    title="Switch > Stay (-log10 p, permutation max-t)",
)
plt.show()

if "logp_max_size" in out_dict:
    plot_glass_brain(
        out_dict["logp_max_size"],
        colorbar=True,
        display_mode="lyrz",
        plot_abs=False,
        title="Switch > Stay (-log10 p, permutation cluster size)",
    )
    plt.show()

## 7. Save Group-Level Results

In [ ]:
output_dir = Path("../results")

z_map_group.to_filename(str(output_dir / "group_switch-vs-stay_zmap.nii.gz"))
print("Group z-map saved.")

neg_log_pval.to_filename(str(output_dir / "group_switch-vs-stay_bonferroni_neglog10p.nii.gz"))
print("Bonferroni p-value map saved.")

out_dict["logp_max_t"].to_filename(str(output_dir / "group_switch-vs-stay_permutation_logp.nii.gz"))
print("Permutation test map saved.")